# Data Preparation

This section gathers all of the work that turns the raw source files into the analytical dataset used by the chapters that follow. It proceeds in two stages: 

- The first stage cleans the chemical data, which arrives in two raw files with inconsistent formats and unit conventions.
  
- The second stage merges the cleaned chemical data with the environmental and taxa datasets, retaining only sites with complete records across all three sources.

---

## Stage 1: Chemical Data Cleaning

**Objectives**

This stage turns two raw chemical datasets — collected in different years, recorded in inconsistent formats, and carrying a number of hidden issues — into one clean chemical table where every value is comparable across sites and ready for downstream merging.

Specifically, the two raw files separately store different parts of the chemical data and adopt different unit conventions for some chemicals. One of them additionally requires fixing a mixture of two unit conventions within a single column.

- **major chemical dataset**: 245 sites, 16 chemicals, values on $\log_2(Y+1)$ scale, units not recorded but inferable, Hg and Zn columns mixing two unit conventions.

- **1991 chemical dataset**: 77 sites, 16 chemicals, units already consistent per column but several chemical columns contain missing values that need to be filled before merging.

**Reasons**

The chemical data are spread across two source files that were prepared at different times by different people, and the raw files cannot be used directly. One file stores values on a transformed scale rather than the original measurements. Neither file records measurement units, even though different chemicals are clearly measured in different ways. A few columns mix two unit conventions within the same column. The other file has missing values that need to be filled before any site-level comparison can be made. Together, these issues mean that if the two files were simply stacked together as-is, the same chemical at the same concentration could appear as wildly different numbers depending on which year the sample came from — making any cross-site comparison meaningless.

**How**

The two files are cleaned separately first. Each one goes through its own set of fixes appropriate to its specific issues — back-transforming scaled values, inferring missing units, normalizing inconsistent units within columns, filling missing values, recording final units. Once both files are individually clean, their unit conventions are compared and aligned to a single target unit per chemical, duplicate rows that appear in both files are removed, and the two are concatenated into one chemical dataset.

**Contributions**

- A cleaned **major chemical dataset** of 245 sites with internally consistent units across all 16 chemicals.
- A cleaned **1991 chemical dataset** of 77 sites with missing values filled and units recorded.
- A merged **chemical dataset** of 322 sites, row-unique and fully aligned on a single universal unit array across all 16 chemicals.

---

#### Step 1: Clean the major chemical dataset

**Inputs**

- file: `log21P_MarjorChe_Sites.xlsx` (raw chemical dataset, third sheet) — values stored on $\log_2(Y+1)$ scale, units not recorded, Hg and Zn columns mixing two unit conventions.

**Process**

*Transformations*:

- *missing-row remover and column retainer*

  - It scans the third sheet of the raw file, drops rows where every chemical column is missing (these are the 1991-labeled rows in this file), and keeps the 16 targeted chemical columns alongside the site IDs and relevant sample information.

- *log2-plus-one back-transformer*

  - It accepts a value $y$ on the $\log_2(Y+1)$ scale and returns the original measurement $Y = 2^{y} - 1$ for every cell in the chemical columns.

*Bridging tools*:

- *unit inferrer*

  - It accepts the back-transformed chemical values and infers the original measurement unit of each chemical by comparing the magnitude of values to the typical scale at which that chemical is reported — trace elements appear on smaller scales, bulk compounds on larger ones.

- *internal unit uniformer*

  - It accepts the **Hg** and **Zn** columns and the year label of each sample. Within these two columns the unit convention shifted between sampling years: the 2004–2005 Hg values are in ug/g while the rest are in ng/g; the 1999 Zn values are in mg/g while the rest are in ug/g. The uniformer applies the year-conditional conversions

  $$\text{Hg}_{\text{ug/g}} \times 1000 = \text{Hg}_{\text{ng/g}}; \qquad \text{Zn}_{\text{mg/g}} \times 1000 = \text{Zn}_{\text{ug/g}}$$

  so that each column ends up internally consistent in one unit.

- *unit array recorder*

  - It accepts the cleaned chemical columns and stores the final unit for each of the 16 chemicals as a unit array attached to the dataset.

**Outputs**

- table: cleaned major chemical dataset
  - rows: $(\text{site ID}, \text{sample info}, \text{chemical}_1, \dots, \text{chemical}_{16})$
  - shape: $245 \times 22$
  - units: per-column unit array; mixed across chemicals (some ug/g, some ng/g), but internally consistent within each column

---

#### Step 2: Clean the 1991 chemical dataset

**Inputs**

- file: `1991Che_DRSites.xls` (raw 1991 chemical dataset) — units already consistent per column, but several chemical columns contain missing values that need to be filled before merging.

**Process**

*Transformations*:

- *column retainer*

  - It selects the same 16 targeted chemical columns and site IDs / sample information as in Step 1, discarding the rest.

*Bridging tools*:

- *detection-limit-based missing filler*

  - It accepts a chemical column with missing values and that chemical's detection limit $X$, then fills every missing cell with a value drawn uniformly from the interval $[0.01X, \, 0.5X]$. Applied to **Hg** (1 missing), **SumPCBs** (45), **Cd** (2), **OCS** (63), and **p,p'-DDE** (52). The detection-limit-based filling approach follows Jian's procedure (2008, p. 31).

- *unit array recorder*

  - It accepts the cleaned chemical columns and stores the unit of each chemical. The 1991 dataset's unit array is **not the same** as the major dataset's unit array, so recording it explicitly is required before any merging can be attempted.

**Outputs**

- table: cleaned 1991 chemical dataset
  - rows: $(\text{site ID}, \text{sample info}, \text{chemical}_1, \dots, \text{chemical}_{16})$
  - shape: $77 \times 22$
  - units: per-column unit array; internally consistent within each column, but distinct from the major dataset's units for several chemicals

---

#### Step 3: Merge the two cleaned chemical datasets

**Inputs**

- table: cleaned major chemical dataset $(245 \times 22)$ from Step 1
- table: cleaned 1991 chemical dataset $(77 \times 22)$ from Step 2

**Process**

*Bridging tools*:

- *universal unit converter*

  - It accepts the two cleaned datasets together with their respective unit arrays and a target unit array, then converts each chemical column elementwise so that both datasets share the same unit per chemical. The target unit array is set chemical-by-chemical based on which scale is the more interpretable.

- *cross-dataset duplicate remover*

  - It scans the cleaned major chemical dataset and removes any rows already present in the cleaned 1991 dataset. The major dataset's 1991-labeled samples are a subset of those in the dedicated 1991 file, so removing the overlap leaves 233 unique major-dataset rows from 1999, 2004, and 2005.

- *row-wise concatenator*

  - It accepts the two unit-aligned datasets and stacks them row-wise. No common key is required because the two datasets share identical column structure and units; the operation adds more sites of the same data type rather than enriching individual sites with new variables.

- *unit array recorder*

  - It accepts the merged dataset and stores the universal target unit array as the final record of measurement units, since unit interpretation determines how downstream transformations should behave.

**Outputs**

- table <span style="color: blue;">C1</span>: clean merged chemical dataset
  - rows: $(\text{site ID}, \text{sample info}, \text{chemical}_1, \dots, \text{chemical}_{16})$
  - shape: $322 \times 22$
  - units: universal target unit array (see table below)

The universal target unit array, with rows where the two source datasets disagreed highlighted in yellow:

<div align="center">

<table style="font-size: 12px; text-align: center;">
  <thead>
    <tr>
      <th>Chemical</th>
      <th>Major Set Units</th>
      <th>1991 DR Set Units</th>
      <th>Units in Jian's Table</th>
      <th>Target Units</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Co</td><td>ug/g</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr style="background-color: #fff3b0;">
      <td>Al</td><td>ug/g</td><td>mg/g</td><td>mg/g</td><td>mg/g</td>
    </tr>
    <tr><td>Ni</td><td>ug/g</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>Mn</td><td>ug/g</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr style="background-color: #fff3b0;">
      <td>Fe</td><td>ug/g</td><td>mg/g</td><td>mg/g</td><td>mg/g</td>
    </tr>
    <tr><td>Cr</td><td>ug/g</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>Cu</td><td>ug/g</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr style="background-color: #fff3b0;">
      <td>Hg</td><td>ng/g or ug/g</td><td>ng/g</td><td>ug/g</td><td>ng/g</td>
    </tr>
    <tr><td>Pb</td><td>ug/g</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr style="background-color: #fff3b0;">
      <td>Zn</td><td>ug/g or mg/g</td><td>ug/g</td><td>ug/g</td><td>ug/g</td>
    </tr>
    <tr><td>SumPCBs</td><td>ng/g</td><td>ng/g</td><td>ng/g</td><td>ng/g</td></tr>
    <tr><td>Cd</td><td>ug/g</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>OCS</td><td>ng/g</td><td>ng/g</td><td>ng/g</td><td>ng/g</td></tr>
    <tr><td>p,p'-DDE</td><td>ng/g</td><td>ng/g</td><td>ng/g</td><td>ng/g</td></tr>
    <tr><td>As</td><td>ug/g</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr style="background-color: #fff3b0;">
      <td>Ca</td><td>ug/g</td><td>mg/g</td><td>mg/g</td><td>mg/g</td>
    </tr>
  </tbody>
</table>

</div>

- Yellow rows indicate where variable units were inconsistent across data sets. They have been fixed by being converted to the target unit.
- Unit equivalences: $1 \text{ mg/g} = 1000 \text{ ug/g}$, $1 \text{ ug/g} = 1000 \text{ ng/g}$.
- Target units are consistent with units of DR 1991 dataset.
- Third column shows units recorded in table 2.13 in Jian's thesis of 2008. **But** it is unclear whether the **unit mixture** issue in Hg and Zn were also existing and fixed in that time. Additionally, the software steps were **not** recorded, which makes it hard to validate if the computation really takes the same units as the table. Therefore, these units are worth reference but can **not** guarantee to reproduce the same results.

---

## Stage 2: Three-Source Merging

**Objectives**

This stage merges the three data sources — environmental, taxa, and the cleaned chemical dataset from Stage 1 — into a single site-wise table containing only the sites with complete data across all three types, ready for the chapter analyses.

**Reasons**

The three datasets each describe a different aspect of the same set of sites: environmental conditions, biological community composition, and chemical contamination. Any chapter analysis that asks how these aspects relate to one another needs them aligned per site. The datasets do not all share the same site identifier, and they do not all contain the same sites — some sites appear in two of the three but not the third. To produce a defensible analytical dataset, the three sources need to be joined through their common keys and reduced to the subset of sites that have complete information in all three.

**How**

The environmental and chemical datasets share a `StationID` column. The taxa and chemical datasets share an `Integrated Code` column. The chemical dataset is therefore the bridge between the other two — it can be joined leftward to the environmental data through `StationID` and rightward to the taxa data through `Integrated Code`. After joining, only sites with complete records in all three are retained, producing the final analytical dataset that downstream chapters consume.

**Contributions**

- A single merged dataset of 310 sites carrying environmental, chemical, and taxa data side by side.
- A documented unit and scale convention per data type, so downstream chapters can apply transformations with full awareness of how each variable was originally measured.

---

#### Step 1: Merge environmental, taxa, and chemical datasets on common keys

**Inputs**

- table: environmental dataset
  - shape: $323 \times (5 + 6)$
  - units: per-column unit array of length 6
- table: taxa dataset
  - shape: $311 \times (5 + 16)$
  - units: octave-scale transformation of relative abundances (Gauch, 1982)
- table <span style="color: blue;">C1</span>: clean merged chemical dataset from Stage 1
  - shape: $322 \times (6 + 16)$
  - units: universal target unit array of length 16

**Process**

*Bridging tools*:

- *common-key joiner*

  - It accepts the three tables and joins them through the chain

  $$\text{environmental} \xleftrightarrow{\text{StationID}} \text{chemical} \xleftrightarrow{\text{Integrated Code}} \text{taxa}$$

  using the chemical table as the bridge. Each site row in the result carries environmental, chemical, and taxa columns side by side.

- *complementary sample-info filler*

  - It accepts the three tables and resolves shared sample information (year, waterbody, etc.) by looking across all three rather than relying on one source: if a site is missing year information in one table but has it recorded in another, the available value is used. Each piece of sample information is stored only once in the merged result.

- *complete-row retainer*

  - It accepts the joined table and retains only the rows that have complete data in all three data types, dropping any site that is missing from one or more of the source tables.

**Outputs**

- table <span style="color: blue;">D1</span>: merged complete-data dataset across the three data types
  - rows: $(\text{site ID, sample info}, \mathbf{E}_i, \mathbf{M}_i, \mathbf{T}_i)$
  - shape: $310 \times (5 + 6 + 16 + 16)$
  - units: environmental column units; chemical universal target unit array; taxa values on octave-scale

The three data matrices used throughout the chapters are extracted from table D1 as follows:

| Data Matrix Source | Symbol | Dimension | Units |
|---|---|---|---|
| Environmental data | $\mathbf{E}$ | $310 \times 6$ | unit array $(6, 1)$ |
| Chemical data (material composition) | $\mathbf{M}$ | $310 \times 16$ | unit array $(16, 1)$ |
| Taxa data (relative abundances) | $\mathbf{T}$ | $310 \times 16$ | octave format |

The taxa values are stored in **octave-scale transformation** of relative abundances, computed from the proportion $p_{ij}$ of taxon $j$ at site $i$ as

$$
y_{ij} = \log_{2}\left[100\,(p_{ij} + 0.01)\right]
$$

This format supports inferring the relative abundance $p_{ij}$ of taxon $j$ at site $i$, but does **not** support recovering the raw absolute abundances.

## Cleaned Data Sets for Two Case Studies

There are two case studies in the thesis. 

Corrior Case study consists of 310 sites spreading a wider area, including the St. Clair River, St. Clair Lake and Detroit River. 

Detroir River case study consists of 213 sites only in the Detroit River.
Its data is a subset of the corridor case data, but in the environmental data $\mathbf{E}$, the DR case has one more variable - Water Velocity.

Below are the $2 \times 3$ specific data sets for every type of data in the two case studies.

## Corridor Case Study

| Data Matrix Source | Symbol | Dimension | Units |
|---|---|---|---|
| Environmental data | $\mathbf{E}$ | $310 \times 6$ | unit array $(6, 1)$ |
| Chemical data (material composition) | $\mathbf{M}$ | $310 \times 16$ | unit array $(16, 1)$ |
| Taxa data (relative abundances) | $\mathbf{T}$ | $310 \times 16$ | octave format |

Links:

- [Corridor_Env_Data.xlsx](data/case_study_full_data/Corridor_Env_Data.xlsx)
- [Corridor_Chemical_Data.xlsx](data/case_study_full_data/Corridor_Chemical_Data.xlsx)
- [Corridor_Taxa_Data.xlsx](data/case_study_full_data/Corridor_Taxa_Data.xlsx)

- Note: water velocity values are not available for whole corridor sites, so only 5 of the 6 environmental variables are used in the corridor case study.


## Detroit River Case Study

| Data Matrix Source | Symbol | Dimension | Units |
|---|---|---|---|
| Environmental data | $\mathbf{E}$ | $213 \times 6$ | unit array $(6, 1)$ |
| Chemical data (material composition) | $\mathbf{M}$ | $213 \times 16$ | unit array $(16, 1)$ |
| Taxa data (relative abundances) | $\mathbf{T}$ | $213 \times 16$ | octave format |

Links:

- [DR_Env_Data.xlsx](data/case_study_full_data/DR_Env_Data.xlsx)
- [DR_Chemical_Data.xlsx](data/case_study_full_data/DR_Chemical_Data.xlsx)
- [DR_Taxa_Data.xlsx](data/case_study_full_data/DR_Taxa_Data.xlsx)

- Note: water velocity values are available for all Detroit River sites, so all 6 environmental variables are used in the Detroit River case study.